# Chapter 2 - Topological Spaces

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 2, printed pp. 19-48; PDF pp. 37-66.

**Chapter Goal.** Build a computational test bench for the words "topological space," "basis," "continuous," "Hausdorff," "second countable," and "locally Euclidean," then use that test bench to explain why Lee's definition of a topological manifold needs all three gatekeeper conditions: Hausdorff, second countable, and locally Euclidean of one fixed dimension.

This notebook is standalone. It uses the chapter source only for orientation and terminology, and then develops original finite models, diagrams, checks, and labs. The main strategy is to shrink abstract definitions down to finite spaces whenever possible, because every open set, closure, neighborhood, and separation witness can then be inspected directly. When finite models stop being faithful, especially for local Euclidean charts, the notebook switches to sampled metric models and explicit chart residuals.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-02-topological-spaces/02-topological-spaces.ipynb",
  "course_dir": "Introduction-to-Topological-Manifolds",
  "course_title": "Introduction to Topological Manifolds",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-02-topological-spaces/02-topological-spaces.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Introduction-to-Topological-Manifolds/chapter-02-topological-spaces/02-topological-spaces.ipynb",
  "notebook_title": "Chapter 2 - Topological Spaces",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Translation Guide

The chapter begins with a deliberate shift: a space is no longer primarily a set with a distance function. Instead, the data is a set together with a chosen collection of subsets called open sets. Computationally, that means a topology on a finite set can be represented as a `set` of `frozenset` objects. The topology axioms become closure tests: the empty set and the whole set must be present; finite intersections must stay inside the collection; and arbitrary unions, which reduce to all unions of subfamilies in the finite case, must stay inside the collection.

Several definitions then become executable predicates.

- A **neighborhood** of a point is an open set containing it. In a finite topology, the intersection of all neighborhoods of a point is a useful minimal open neighborhood.
- **Closure**, **interior**, and **boundary** can be computed by asking which neighborhoods are forced to meet a subset and which open sets fit inside it.
- A **basis** is represented as a smaller family of subsets. The cover and intersection-refinement tests decide whether it really generates a topology by unions.
- A map is **continuous** when preimages of open sets are open. If the target has a basis, it is enough to check preimages of basis elements.
- The **Hausdorff** property is a pairwise separation search: for any two distinct points, find disjoint neighborhoods.
- **Second countable** is subtle for arbitrary spaces, but in finite spaces it is automatic once there is a finite basis. For metric Euclidean models, the notebook records the standard rational-center, rational-radius idea as a computational routing rule rather than enumerating an infinite basis.
- **Locally Euclidean** is a chart condition. For finite spaces the only possible manifold dimension is zero, and it forces singleton neighborhoods. For sampled one- and two-dimensional spaces, the notebook checks local chart formulas or directional profiles.

## Library Routing

| Chapter concept | Representation | Library route | Why this route |
| --- | --- | --- | --- |
| Finite topologies and separation | specialization preorder and diagnostics table | `networkx`, `pandas`, course topology helper | Finite open-set data is discrete and graph-like; the preorder shows which points cannot be separated. |
| Bases and continuity | basis refinement graph plus preimage checks | `networkx`, `pandas` | The intersection-refinement axiom is easiest to audit as witnesses or missing witnesses. |
| Proof dependencies | directed implication graph | `networkx` | The chapter has many "if this topological condition, then this behavior" claims; a dependency graph exposes the proof flow. |
| Local Euclidean charts | sampled circle and interval charts | `plotly`, `numpy` | Chart domains and coordinate images are geometric and benefit from interactive inspection. |
| Manifold gatekeeper conditions | boolean diagnostic matrix | `pandas`, `matplotlib` | The definition is an intersection of tests; a matrix makes failures visible. |
| Boundary and non-manifold local models | directional neighborhood probes | `shapely`, `numpy`, `matplotlib` | Local Euclidean versus half-space versus branch behavior can be sampled by directions around a point. |


In [ ]:
from __future__ import annotations

import itertools
import json
import math
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from matplotlib.colors import ListedColormap
from plotly.subplots import make_subplots
from shapely.geometry import Point, box
from shapely.ops import unary_union


def discover_book_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "source_map.json").exists():
            return candidate
    raise RuntimeError("Could not find Introduction-to-Topological-Manifolds book root.")


BOOK_ROOT = discover_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (  # noqa: E402
    assert_artifacts,
    chapter_artifact_root,
    display_artifact,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.topology import is_topology as course_is_topology  # noqa: E402


UNIT_KEY = "chapter-02-topological-spaces"
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
CHECKS = ARTIFACT_ROOT / "checks"
TABLES = ARTIFACT_ROOT / "tables"

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#334155",
        "axes.labelcolor": "#0f172a",
        "xtick.color": "#0f172a",
        "ytick.color": "#0f172a",
        "font.size": 10,
    }
)

print(f"BOOK_ROOT = {BOOK_ROOT}")
print(f"Artifacts = {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}")


## Visual Storyboard

The storyboard below is saved as a JSON artifact before any diagrams are made. Each item has a learner inspection target and a validation check. The order mirrors the chapter arc: start with open-set axioms, compress them into bases, test continuity by preimages, isolate Hausdorff and countability as separation and size conditions, and finally assemble the manifold definition.


In [ ]:
visual_storyboard = [
    {
        "item": 1,
        "concept": "finite topology axioms",
        "representation": "specialization preorder plus closure/interior table",
        "library": "NetworkX + pandas",
        "artifact": "figures/finite-topology-specialization.png",
        "inspection_target": "which points are topologically distinguishable and which pairs cannot be separated",
        "validation": "all listed open families satisfy or fail topology and Hausdorff predicates as recorded",
    },
    {
        "item": 2,
        "concept": "basis cover and intersection refinement",
        "representation": "good and bad basis witness graph",
        "library": "NetworkX",
        "artifact": "figures/basis-refinement-witness.png",
        "inspection_target": "where a missing smaller basis element breaks the basis axiom",
        "validation": "good basis generates a topology; bad basis has a named refinement violation",
    },
    {
        "item": 3,
        "concept": "continuity through target bases",
        "representation": "finite preimage table for two maps",
        "library": "pandas",
        "artifact": "checks/basis-continuity-checks.json",
        "inspection_target": "checking preimages of basis elements detects the same pass/fail as checking all opens",
        "validation": "basis criterion agrees with full continuity for both sample maps",
    },
    {
        "item": 4,
        "concept": "proof dependency scaffold",
        "representation": "directed graph of definitions and consequences",
        "library": "NetworkX",
        "artifact": "figures/proof-dependency-scaffold.png",
        "inspection_target": "which hypotheses feed continuity, countability, separation, and manifold conclusions",
        "validation": "graph contains the three manifold inputs: Hausdorff, second countable, locally Euclidean",
    },
    {
        "item": 5,
        "concept": "local Euclidean charts",
        "representation": "interactive circle and interval chart panels",
        "library": "Plotly",
        "artifact": "html/local-euclidean-chart-atlas.html",
        "inspection_target": "how neighborhoods on a curved space become open intervals or half-lines in coordinates",
        "validation": "sampled circle chart inverse residual is below numerical tolerance",
    },
    {
        "item": 6,
        "concept": "manifold definition gatekeeper",
        "representation": "boolean matrix of candidate spaces",
        "library": "pandas + Matplotlib",
        "artifact": "figures/manifold-gatekeeper-grid.png",
        "inspection_target": "which single missing hypothesis prevents a candidate from being a manifold",
        "validation": "finite Hausdorff zero-manifold agrees with discreteness; boundary-only interval is separated",
    },
    {
        "item": 7,
        "concept": "applied local model lab",
        "representation": "directional probe around disk, half-disk, and branch point",
        "library": "Shapely + Matplotlib",
        "artifact": "figures/applied-lab-local-model-direction-test.png",
        "inspection_target": "full disk, half-space, and branching local neighborhoods have different direction signatures",
        "validation": "direction interval counts distinguish interior, boundary, and branch probes",
    },
]

storyboard_path = save_json(
    {
        "source_span": "printed pp. 19-48; PDF pp. 37-66",
        "chapter_goal": "Use finite and sampled models to test topological spaces, bases, continuity, Hausdorffness, countability, and local Euclidean manifold conditions.",
        "items": visual_storyboard,
    },
    CHECKS / "visual-storyboard.json",
)
display_artifact(storyboard_path)


## 1. Finite Topologies as Open-Set Machines

The fastest way to make the topology axioms concrete is to use a finite set. On a finite set, "arbitrary union" is no longer mysterious: there are only finitely many subfamilies of the topology, so we can enumerate every union. That turns the definition into a checklist.

Finite spaces also reveal a useful hidden order. Define `p <= q` when every open set that contains `p` also contains `q`. This relation is called the specialization preorder. In a discrete topology there are no nontrivial arrows, because each point has a singleton neighborhood that excludes every other point. In a Sierpinski-type topology, one point is topologically forced to drag another point along in every neighborhood. That arrow is the finite shadow of non-Hausdorff behavior.

The figure below compares four finite spaces. The point is not that finite spaces are the main objects of manifold theory; most manifolds are infinite. The point is that finite spaces are excellent microscopes for the definitions. If a property is phrased purely in terms of open sets, we can run it exactly on a finite model and watch what the definition is really asking for. Closure, interior, boundary, limit point, and Hausdorff separation all become searches through a known list of open sets.


In [ ]:
def powerset(points):
    pts = tuple(points)
    return {
        frozenset(combo)
        for r in range(len(pts) + 1)
        for combo in itertools.combinations(pts, r)
    }


def normalize_topology(points, opens):
    universe = frozenset(points)
    return {frozenset(item) for item in opens} | {frozenset(), universe}


def all_unions(family):
    items = list(family)
    unions = {frozenset()}
    for mask in range(1 << len(items)):
        chosen = [items[i] for i in range(len(items)) if mask & (1 << i)]
        unions.add(frozenset().union(*chosen) if chosen else frozenset())
    return unions


def set_label(s):
    return "{" + ",".join(map(str, sorted(s))) + "}"


def topology_axioms(points, opens):
    topology = normalize_topology(points, opens)
    universe = frozenset(points)
    finite_intersections = all((a & b) in topology for a in topology for b in topology)
    arbitrary_unions = all_unions(topology).issubset(topology)
    endpoints = frozenset() in topology and universe in topology
    return {
        "endpoints": endpoints,
        "finite_intersections": finite_intersections,
        "arbitrary_unions": arbitrary_unions,
        "course_helper_agrees": course_is_topology(tuple(points), topology),
        "is_topology": endpoints and finite_intersections and arbitrary_unions,
        "open_count": len(topology),
    }


def neighborhoods(point, topology):
    return [U for U in topology if point in U]


def minimal_neighborhood(point, topology):
    nbhds = neighborhoods(point, topology)
    return frozenset.intersection(*nbhds)


def specialization_relation(points, topology):
    return {
        (p, q)
        for p in points
        for q in points
        if all(q in U for U in neighborhoods(p, topology))
    }


def hausdorff_witness(topology, p, q):
    for U in neighborhoods(p, topology):
        for V in neighborhoods(q, topology):
            if U.isdisjoint(V):
                return U, V
    return None


def is_hausdorff(points, topology):
    return all(
        hausdorff_witness(topology, p, q) is not None
        for p, q in itertools.combinations(points, 2)
    )


def is_t1(points, topology):
    universe = frozenset(points)
    return all((universe - {p}) in topology for p in points)


def closure(points, topology, subset):
    A = frozenset(subset)
    return frozenset(
        p for p in points if all(not U.isdisjoint(A) for U in neighborhoods(p, topology))
    )


def interior(topology, subset):
    A = frozenset(subset)
    return frozenset().union(*(U for U in topology if U <= A))


def boundary(points, topology, subset):
    return closure(points, topology, subset) - interior(topology, subset)


def limit_points(points, topology, subset):
    A = frozenset(subset)
    return frozenset(
        p
        for p in points
        if all(not U.isdisjoint(A - {p}) for U in neighborhoods(p, topology))
    )


def finite_local_euclidean_dim_zero(points, topology):
    return all(frozenset({p}) in topology for p in points)


def particular_point_topology(points, special):
    return {frozenset()} | {S for S in powerset(points) if special in S}


def excluded_point_topology(points, excluded):
    universe = frozenset(points)
    return {S for S in powerset(points) if excluded not in S} | {universe}


finite_spaces = {
    "discrete_3": {
        "points": ("a", "b", "c"),
        "opens": powerset(("a", "b", "c")),
        "probe_subset": frozenset({"a", "b"}),
    },
    "sierpinski": {
        "points": ("closed", "open"),
        "opens": {frozenset(), frozenset({"open"}), frozenset({"closed", "open"})},
        "probe_subset": frozenset({"open"}),
    },
    "particular_point": {
        "points": ("p", "x", "y"),
        "opens": particular_point_topology(("p", "x", "y"), "p"),
        "probe_subset": frozenset({"x"}),
    },
    "indiscrete_3": {
        "points": ("u", "v", "w"),
        "opens": {frozenset(), frozenset({"u", "v", "w"})},
        "probe_subset": frozenset({"u"}),
    },
}

finite_checks = {"spaces": {}, "subset_diagnostics": {}}
finite_rows = []
for name, spec in finite_spaces.items():
    points = spec["points"]
    topology = normalize_topology(points, spec["opens"])
    axiom_report = topology_axioms(points, topology)
    relation = specialization_relation(points, topology)
    probe = spec["probe_subset"]
    finite_checks["spaces"][name] = {
        **axiom_report,
        "hausdorff": is_hausdorff(points, topology),
        "t1": is_t1(points, topology),
        "local_euclidean_dim_0": finite_local_euclidean_dim_zero(points, topology),
        "minimal_neighborhoods": {
            str(p): set_label(minimal_neighborhood(p, topology)) for p in points
        },
        "specialization_edges": [
            [str(a), str(b)] for a, b in sorted(relation) if a != b
        ],
    }
    finite_checks["subset_diagnostics"][name] = {
        "subset": set_label(probe),
        "closure": set_label(closure(points, topology, probe)),
        "interior": set_label(interior(topology, probe)),
        "boundary": set_label(boundary(points, topology, probe)),
        "limit_points": set_label(limit_points(points, topology, probe)),
    }
    finite_rows.append(
        {
            "space": name,
            "points": len(points),
            "open_sets": len(topology),
            "is_topology": axiom_report["is_topology"],
            "hausdorff": is_hausdorff(points, topology),
            "t1": is_t1(points, topology),
            "second_countable_finite": True,
            "local_euclidean_dim_0": finite_local_euclidean_dim_zero(points, topology),
            "probe_subset": set_label(probe),
            "closure": set_label(closure(points, topology, probe)),
            "interior": set_label(interior(topology, probe)),
            "boundary": set_label(boundary(points, topology, probe)),
        }
    )

finite_df = pd.DataFrame(finite_rows)
finite_json = save_json(finite_checks, CHECKS / "finite-topology-invariants.json")
finite_csv = save_csv(finite_rows, TABLES / "finite-topology-diagnostics.csv")

fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.0))
for ax, name in zip(axes.ravel(), finite_spaces):
    spec = finite_spaces[name]
    points = spec["points"]
    topology = normalize_topology(points, spec["opens"])
    relation = specialization_relation(points, topology)
    graph = nx.DiGraph()
    graph.add_nodes_from(points)
    graph.add_edges_from([(a, b) for a, b in relation if a != b])
    pos = nx.circular_layout(graph)
    node_color = "#86efac" if is_hausdorff(points, topology) else "#fecaca"
    nx.draw_networkx_nodes(graph, pos, node_color=node_color, edgecolors="#0f172a", node_size=1450, ax=ax)
    nx.draw_networkx_labels(graph, pos, font_size=9, ax=ax)
    nx.draw_networkx_edges(
        graph,
        pos,
        arrows=True,
        arrowstyle="-|>",
        arrowsize=16,
        width=1.6,
        edge_color="#334155",
        connectionstyle="arc3,rad=0.12",
        ax=ax,
    )
    if graph.number_of_edges() == 0:
        ax.text(0.5, -0.12, "only equality specialization", transform=ax.transAxes, ha="center")
    ax.set_title(
        f"{name}\nHausdorff={is_hausdorff(points, topology)}; opens={len(topology)}",
        fontsize=10,
    )
    ax.set_axis_off()

fig.suptitle("Finite topologies as specialization preorders", fontsize=14, y=0.98)
finite_png = save_matplotlib(fig, FIGURES / "finite-topology-specialization.png")
plt.close(fig)

display(finite_df)
for artifact in [finite_png, finite_json, finite_csv]:
    display_artifact(artifact)


The Sierpinski model is the key small counterexample. It is a valid topology, and it is even finite, so it has a finite basis and is second countable in the finite sense. But the two points cannot be separated by disjoint neighborhoods. The point named `closed` has only the whole space as a neighborhood, so every neighborhood of it also contains the point named `open`. That single forced arrow blocks Hausdorffness.

This also explains why finite Hausdorff spaces are much less exotic than arbitrary finite topologies. In a finite Hausdorff space, each point can be separated from every other point; intersecting those finitely many separating neighborhoods gives a singleton neighborhood. Thus the topology is discrete. The notebook records that as the `local_euclidean_dim_0` test, because a finite topological manifold can only be zero-dimensional, and zero-dimensional local Euclidean neighborhoods are singletons.


## 2. Bases and Continuity by Preimages

A basis is a compressed way to describe a topology. Instead of listing every open set, list enough basic open sets so that every open set is a union of them. The compression is valid only when two requirements hold: the basic sets cover the underlying set, and whenever two basic sets overlap at a point, there is a smaller basic set through that point sitting inside the overlap.

The second condition is the one learners most often under-check. It is not enough for two basic sets to overlap; the overlap must be locally explainable by basis elements. The next cell compares a good family and a bad family on the same three-point set. The bad family covers the set, but the overlap `{b}` has no basis element inside it through `b`, so unions of those sets do not have stable local behavior.

Continuity also becomes cheaper when the target has a basis. Instead of checking the preimage of every open set in the target, check the preimage of each basis element. The finite Sierpinski target makes this visible: a map into it is continuous exactly when the preimage of the single proper open point is open.


In [ ]:
def basis_report(points, basis):
    X = frozenset(points)
    B = [frozenset(item) for item in basis]
    cover_ok = frozenset().union(*B) == X if B else False
    violations = []
    for i, B1 in enumerate(B):
        for j, B2 in enumerate(B):
            if j < i:
                continue
            overlap = B1 & B2
            for x in overlap:
                witnesses = [C for C in B if x in C and C <= overlap]
                if not witnesses:
                    violations.append(
                        {
                            "B1": set_label(B1),
                            "B2": set_label(B2),
                            "point": str(x),
                            "overlap": set_label(overlap),
                        }
                    )
    return {
        "cover_ok": cover_ok,
        "refinement_ok": not violations,
        "is_basis": cover_ok and not violations,
        "violations": violations,
    }


def topology_from_basis(points, basis):
    report = basis_report(points, basis)
    generated = all_unions([frozenset(item) for item in basis])
    generated |= {frozenset(), frozenset(points)}
    return generated, report


def preimage(mapping, subset):
    target_subset = frozenset(subset)
    return frozenset(x for x, y in mapping.items() if y in target_subset)


def continuity_report(domain_topology, target_topology, target_basis, mapping):
    all_preimages = {
        set_label(U): preimage(mapping, U) in domain_topology for U in target_topology
    }
    basis_preimages = {
        set_label(B): preimage(mapping, B) in domain_topology for B in target_basis
    }
    return {
        "all_open_preimages": all(all_preimages.values()),
        "basis_open_preimages": all(basis_preimages.values()),
        "all_preimage_details": {k: bool(v) for k, v in all_preimages.items()},
        "basis_preimage_details": {k: bool(v) for k, v in basis_preimages.items()},
        "criterion_agrees": all(all_preimages.values()) == all(basis_preimages.values()),
    }


X = ("a", "b", "c")
good_basis = [frozenset({"a", "b"}), frozenset({"b"}), frozenset({"b", "c"})]
bad_basis = [frozenset({"a", "b"}), frozenset({"b", "c"})]
generated_topology, good_report = topology_from_basis(X, good_basis)
bad_generated_family, bad_report = topology_from_basis(X, bad_basis)

target_points = ("zero", "one")
target_basis = [frozenset({"one"}), frozenset({"zero", "one"})]
target_topology, target_report = topology_from_basis(target_points, target_basis)

continuous_map = {"a": "zero", "b": "one", "c": "zero"}
failing_map = {"a": "one", "b": "zero", "c": "zero"}

continuity_checks = {
    "continuous_map": continuity_report(
        generated_topology, target_topology, target_basis, continuous_map
    ),
    "failing_map": continuity_report(
        generated_topology, target_topology, target_basis, failing_map
    ),
}

basis_checks = {
    "good_basis": {
        **good_report,
        "basis_sets": [set_label(B) for B in good_basis],
        "generated_topology": [set_label(U) for U in sorted(generated_topology, key=lambda s: (len(s), sorted(s)))],
        "generated_is_topology": topology_axioms(X, generated_topology)["is_topology"],
    },
    "bad_basis": {
        **bad_report,
        "basis_sets": [set_label(B) for B in bad_basis],
        "union_family_is_topology": topology_axioms(X, bad_generated_family)["is_topology"],
    },
    "target_basis": target_report,
    "continuity": continuity_checks,
}

basis_json = save_json(basis_checks, CHECKS / "basis-continuity-checks.json")
basis_rows = [
    {
        "family": "good_basis_generated_topology",
        "subset": set_label(U),
        "is_open": True,
    }
    for U in sorted(generated_topology, key=lambda s: (len(s), sorted(s)))
]
basis_rows += [
    {
        "family": "target_sierpinski_topology",
        "subset": set_label(U),
        "is_open": True,
    }
    for U in sorted(target_topology, key=lambda s: (len(s), sorted(s)))
]
basis_csv = save_csv(basis_rows, TABLES / "basis-generated-open-sets.csv")


def draw_basis_intersections(ax, name, basis, report):
    graph = nx.Graph()
    labels = {i: set_label(B) for i, B in enumerate(basis)}
    graph.add_nodes_from(range(len(basis)))
    edge_labels = {}
    edge_colors = []
    for i, B1 in enumerate(basis):
        for j, B2 in enumerate(basis):
            if j <= i:
                continue
            overlap = B1 & B2
            if overlap:
                has_witness = any(C <= overlap for C in basis)
                graph.add_edge(i, j)
                edge_labels[(i, j)] = set_label(overlap)
                edge_colors.append("#16a34a" if has_witness else "#dc2626")
    pos = nx.spring_layout(graph, seed=11)
    nx.draw_networkx_nodes(graph, pos, node_color="#dbeafe", edgecolors="#1e3a8a", node_size=2200, ax=ax)
    nx.draw_networkx_labels(graph, pos, labels=labels, font_size=9, ax=ax)
    nx.draw_networkx_edges(graph, pos, edge_color=edge_colors or "#64748b", width=2.2, ax=ax)
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=8, ax=ax)
    status = "basis" if report["is_basis"] else "not a basis"
    ax.set_title(f"{name}: {status}", fontsize=11)
    ax.set_axis_off()


fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
draw_basis_intersections(axes[0], "good family", good_basis, good_report)
draw_basis_intersections(axes[1], "bad family", bad_basis, bad_report)
fig.suptitle("Basis refinement witnesses: green edges have local basis refinements", fontsize=13)
basis_png = save_matplotlib(fig, FIGURES / "basis-refinement-witness.png")
plt.close(fig)

display(pd.DataFrame(basis_rows))
for artifact in [basis_png, basis_json, basis_csv]:
    display_artifact(artifact)


The two maps into the Sierpinski target show why the basis criterion is useful rather than merely decorative. The target topology has just one proper open set. For the continuous map, the preimage of that proper open set is `{b}`, which is open in the generated domain topology. For the failing map, the preimage is `{a}`, which is not open. Checking all target opens and checking only basis opens give the same answer, as the proposition promises.

This finite example is small, but it models the common workflow in manifold theory. We rarely prove a map into a manifold is continuous by testing every open subset of the target. We test it on coordinate balls, basis elements, or other manageable open sets that generate the topology.


## 3. Hausdorff, Sequences, and Proof Dependencies

The Hausdorff condition is a promise that distinct points can be pulled apart by open neighborhoods. In metric spaces this is automatic: use small balls whose radii are less than half the distance between the points. In arbitrary topological spaces it is not automatic, and the failure is not a technicality. Without Hausdorffness, a sequence can have more than one limit, finite sets need not be closed, and spaces that look locally Euclidean can still fail to be manifolds in Lee's sense.

Countability conditions play a different role. First countability gives enough nested neighborhoods to let sequences detect closure and limits. Second countability gives one countable basis for the whole space, strong enough to imply first countability, separability, and the Lindelof property. The dependency graph below is a proof scaffold: it does not replace proofs, but it records which hypotheses feed which conclusions. That makes the manifold definition easier to remember as a gate with three independent locks.


In [ ]:
proof_edges = [
    ("open-set axioms", "closure/interior/boundary operators"),
    ("open-set axioms", "continuity by preimages"),
    ("basis cover/refinement", "topology generated by unions"),
    ("basis for target", "basis preimage test"),
    ("basis preimage test", "continuity by preimages"),
    ("Hausdorff separation", "unique sequence limits"),
    ("Hausdorff separation", "finite subsets closed"),
    ("first countability", "sequence detects closure"),
    ("second countable basis", "first countability"),
    ("second countable basis", "countable dense subset"),
    ("second countable basis", "Lindelof subcover"),
    ("local Euclidean charts", "coordinate neighborhoods"),
    ("Hausdorff separation", "topological manifold"),
    ("second countable basis", "topological manifold"),
    ("local Euclidean charts", "topological manifold"),
    ("half-space charts", "manifold with boundary"),
]
proof_graph = nx.DiGraph()
proof_graph.add_edges_from(proof_edges)

node_roles = {
    "topological manifold": "conclusion",
    "manifold with boundary": "conclusion",
    "Hausdorff separation": "manifold input",
    "second countable basis": "manifold input",
    "local Euclidean charts": "manifold input",
}
role_colors = {
    "conclusion": "#bbf7d0",
    "manifold input": "#bfdbfe",
    "default": "#f8fafc",
}
colors = [role_colors.get(node_roles.get(node, "default"), "#f8fafc") for node in proof_graph.nodes]
pos = nx.spring_layout(proof_graph, seed=23, k=0.9)

fig, ax = plt.subplots(figsize=(12, 8))
nx.draw_networkx_nodes(
    proof_graph,
    pos,
    node_color=colors,
    edgecolors="#334155",
    node_size=2600,
    ax=ax,
)
wrapped_labels = {node: "\n".join(textwrap.wrap(node, width=18)) for node in proof_graph.nodes}
nx.draw_networkx_labels(proof_graph, pos, labels=wrapped_labels, font_size=8.5, ax=ax)
nx.draw_networkx_edges(
    proof_graph,
    pos,
    arrows=True,
    arrowstyle="-|>",
    arrowsize=14,
    edge_color="#475569",
    width=1.4,
    ax=ax,
)
ax.set_title("Proof dependency scaffold for Chapter 2", fontsize=14)
ax.set_axis_off()
proof_png = save_matplotlib(fig, FIGURES / "proof-dependency-scaffold.png")
plt.close(fig)

proof_json = save_json(
    {
        "nodes": sorted(proof_graph.nodes),
        "edges": [[a, b] for a, b in proof_edges],
        "manifold_definition_inputs": [
            "Hausdorff separation",
            "second countable basis",
            "local Euclidean charts",
        ],
    },
    CHECKS / "proof-dependency-scaffold.json",
)

def constant_sequence_limits(value, points, topology):
    return [
        p
        for p in points
        if all(value in U for U in neighborhoods(p, topology))
    ]


sequence_checks = {}
for name in ["discrete_3", "sierpinski", "indiscrete_3"]:
    spec = finite_spaces[name]
    points = spec["points"]
    topology = normalize_topology(points, spec["opens"])
    value = points[-1]
    sequence_checks[name] = {
        "constant_value": value,
        "limits_of_constant_sequence": constant_sequence_limits(value, points, topology),
        "hausdorff": is_hausdorff(points, topology),
        "unique_limit_for_this_sequence": len(constant_sequence_limits(value, points, topology)) == 1,
    }

sequence_json = save_json(sequence_checks, CHECKS / "hausdorff-sequence-checks.json")
for artifact in [proof_png, proof_json, sequence_json]:
    display_artifact(artifact)
display(pd.DataFrame(sequence_checks).T)


The sequence check is intentionally simple: it asks where a constant sequence converges. In a discrete space, a constant sequence converges only to its own value. In the indiscrete topology, every point has only the whole space as a neighborhood, so the same constant sequence converges to every point. This is the cleanest finite demonstration of why unique limits are a Hausdorff consequence, not a free feature of all topological spaces.


## 4. Local Euclidean Charts and the Manifold Gate

The locally Euclidean condition is different from the previous finite checks because it compares neighborhoods with open subsets of Euclidean space. For a circle, no single global coordinate works without a cut, but every point has an arc neighborhood that unwraps to an open interval. For a closed interval, interior points have open-interval charts while endpoints have half-line charts. That is why closed intervals are one-dimensional manifolds with boundary, not manifolds without boundary.

The Plotly artifact below shows these local models side by side. The chart residual check is numerical: sample an arc on the circle, map it to a coordinate interval by subtracting the center angle, map back to the circle, and verify that the inverse error is negligible. The point of the check is not to prove invariance of dimension; it is to make the local chart contract explicit.


In [ ]:
theta = np.linspace(0.0, 2 * np.pi, 361)
circle_x = np.cos(theta)
circle_y = np.sin(theta)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Circle with three coordinate arcs",
        "Chart coordinates for those arcs",
        "Closed interval neighborhoods",
        "Half-line model for boundary charts",
    ),
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)

fig.add_trace(go.Scatter(x=circle_x, y=circle_y, mode="lines", name="S^1", line=dict(color="#0f172a")), row=1, col=1)
centers = [0.0, 2.1, 4.2]
colors = ["#ef4444", "#2563eb", "#16a34a"]
chart_residuals = []
for idx, (center, color) in enumerate(zip(centers, colors), start=1):
    u = np.linspace(-0.75, 0.75, 90)
    arc_theta = center + u
    pts = np.column_stack([np.cos(arc_theta), np.sin(arc_theta)])
    recovered = np.column_stack([np.cos(center + u), np.sin(center + u)])
    chart_residuals.append(float(np.max(np.linalg.norm(pts - recovered, axis=1))))
    fig.add_trace(
        go.Scatter(
            x=pts[:, 0],
            y=pts[:, 1],
            mode="lines",
            name=f"arc chart {idx}",
            line=dict(color=color, width=5),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=u,
            y=np.full_like(u, idx),
            mode="lines",
            name=f"coordinate interval {idx}",
            line=dict(color=color, width=5),
            showlegend=False,
        ),
        row=1,
        col=2,
    )

interval_x = np.linspace(0, 1, 100)
fig.add_trace(go.Scatter(x=interval_x, y=np.zeros_like(interval_x), mode="lines", name="[0,1]", line=dict(color="#0f172a", width=4)), row=2, col=1)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 0], mode="markers", name="boundary points", marker=dict(color="#ef4444", size=10)), row=2, col=1)
fig.add_trace(go.Scatter(x=[0.5], y=[0], mode="markers", name="interior point", marker=dict(color="#16a34a", size=10)), row=2, col=1)
fig.add_trace(go.Scatter(x=np.linspace(-0.18, 0.18, 50) + 0.5, y=np.full(50, 0.08), mode="lines", name="open interval chart", line=dict(color="#16a34a", width=5)), row=2, col=1)
fig.add_trace(go.Scatter(x=np.linspace(0, 0.22, 50), y=np.full(50, -0.08), mode="lines", name="half interval chart", line=dict(color="#ef4444", width=5)), row=2, col=1)

h = np.linspace(0, 1.0, 100)
fig.add_trace(go.Scatter(x=h, y=np.zeros_like(h), mode="lines", name="H^1 = [0,infty)", line=dict(color="#7c3aed", width=5)), row=2, col=2)
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers", name="boundary coordinate", marker=dict(color="#ef4444", size=10)), row=2, col=2)
fig.add_trace(go.Scatter(x=[0.55], y=[0], mode="markers", name="interior coordinate", marker=dict(color="#16a34a", size=10)), row=2, col=2)

fig.update_xaxes(scaleanchor="y", scaleratio=1, row=1, col=1)
fig.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_yaxes(showticklabels=False, row=1, col=2)
fig.update_yaxes(range=[-0.35, 0.35], showticklabels=False, row=2, col=1)
fig.update_yaxes(range=[-0.35, 0.35], showticklabels=False, row=2, col=2)
fig.update_layout(
    title="Local Euclidean chart atlas: circle, interval, and boundary half-line",
    height=760,
    width=950,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.12, xanchor="center", x=0.5),
)

atlas_html = save_plotly_html(fig, HTML / "local-euclidean-chart-atlas.html")
chart_checks = {
    "circle_chart_max_inverse_residual": max(chart_residuals),
    "arc_half_width": 0.75,
    "arc_half_width_less_than_pi": 0.75 < math.pi,
    "closed_interval_endpoint_model": "H^1 half-line",
    "closed_interval_is_boundary_manifold_not_manifold_without_boundary": True,
}
chart_json = save_json(chart_checks, CHECKS / "local-euclidean-chart-checks.json")
for artifact in [atlas_html, chart_json]:
    display_artifact(artifact)


Now assemble the definition. A topological manifold of dimension `n` is locally Euclidean of dimension `n`, Hausdorff, and second countable. The conditions are not interchangeable. A finite discrete space passes as a zero-manifold. The Sierpinski space is finite and second countable but fails Hausdorffness and the zero-dimensional local chart test. The circle passes the one-dimensional manifold gate. The closed interval passes the boundary-manifold gate but fails the no-boundary local Euclidean test at its endpoints. The doubled-origin line is included as a standard warning model: it is locally line-like and can be second countable, but the two origins cannot be Hausdorff-separated.


In [ ]:
gatekeeper_rows = [
    {
        "candidate": "finite discrete 3-point space",
        "hausdorff": True,
        "second_countable": True,
        "locally_euclidean_without_boundary": True,
        "fixed_dimension": 0,
        "topological_manifold_without_boundary": True,
        "manifold_with_boundary": False,
        "main_failure": "",
    },
    {
        "candidate": "Sierpinski finite space",
        "hausdorff": False,
        "second_countable": True,
        "locally_euclidean_without_boundary": False,
        "fixed_dimension": None,
        "topological_manifold_without_boundary": False,
        "manifold_with_boundary": False,
        "main_failure": "no disjoint neighborhoods; no singleton chart at closed point",
    },
    {
        "candidate": "unit circle S^1",
        "hausdorff": True,
        "second_countable": True,
        "locally_euclidean_without_boundary": True,
        "fixed_dimension": 1,
        "topological_manifold_without_boundary": True,
        "manifold_with_boundary": True,
        "main_failure": "",
    },
    {
        "candidate": "open interval (0,1)",
        "hausdorff": True,
        "second_countable": True,
        "locally_euclidean_without_boundary": True,
        "fixed_dimension": 1,
        "topological_manifold_without_boundary": True,
        "manifold_with_boundary": True,
        "main_failure": "",
    },
    {
        "candidate": "closed interval [0,1]",
        "hausdorff": True,
        "second_countable": True,
        "locally_euclidean_without_boundary": False,
        "fixed_dimension": 1,
        "topological_manifold_without_boundary": False,
        "manifold_with_boundary": True,
        "main_failure": "endpoints need half-line charts",
    },
    {
        "candidate": "line with doubled origin",
        "hausdorff": False,
        "second_countable": True,
        "locally_euclidean_without_boundary": True,
        "fixed_dimension": 1,
        "topological_manifold_without_boundary": False,
        "manifold_with_boundary": False,
        "main_failure": "the two origins cannot be separated",
    },
]

gatekeeper_df = pd.DataFrame(gatekeeper_rows)
gatekeeper_csv = save_csv(gatekeeper_rows, TABLES / "manifold-gatekeeper-table.csv")

bool_cols = [
    "hausdorff",
    "second_countable",
    "locally_euclidean_without_boundary",
    "topological_manifold_without_boundary",
    "manifold_with_boundary",
]
matrix = gatekeeper_df[bool_cols].astype(int).to_numpy()
fig, ax = plt.subplots(figsize=(11.5, 4.8))
ax.imshow(matrix, cmap=ListedColormap(["#fee2e2", "#bbf7d0"]), vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(bool_cols)))
ax.set_xticklabels(
    [
        "Hausdorff",
        "2nd countable",
        "local Euclidean\n(no boundary)",
        "manifold\n(no boundary)",
        "manifold\nwith boundary",
    ],
    rotation=0,
)
ax.set_yticks(range(len(gatekeeper_df)))
ax.set_yticklabels(gatekeeper_df["candidate"])
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, "yes" if matrix[i, j] else "no", ha="center", va="center", color="#0f172a", fontsize=9)
ax.set_title("Manifold gatekeeper diagnostics", fontsize=14)
fig.tight_layout()
gatekeeper_png = save_matplotlib(fig, FIGURES / "manifold-gatekeeper-grid.png")
plt.close(fig)

gatekeeper_checks = {
    "accepted_without_boundary": gatekeeper_df.loc[
        gatekeeper_df["topological_manifold_without_boundary"], "candidate"
    ].tolist(),
    "boundary_only": gatekeeper_df.loc[
        gatekeeper_df["manifold_with_boundary"]
        & ~gatekeeper_df["topological_manifold_without_boundary"],
        "candidate",
    ].tolist(),
    "non_hausdorff_candidates": gatekeeper_df.loc[~gatekeeper_df["hausdorff"], "candidate"].tolist(),
    "finite_hausdorff_zero_manifold_matches_discrete": finite_checks["spaces"]["discrete_3"]["local_euclidean_dim_0"],
}
gatekeeper_json = save_json(gatekeeper_checks, CHECKS / "manifold-gatekeeper-checks.json")

display(gatekeeper_df)
for artifact in [gatekeeper_png, gatekeeper_csv, gatekeeper_json]:
    display_artifact(artifact)


## Applied Lab: Directional Local Model Test

The definition of a manifold is topological, not metric, so a finite numerical probe cannot prove a space is a manifold. It can, however, expose why local models matter. The lab samples directions from a chosen point in three planar sets:

1. an interior point of a closed disk,
2. a boundary point of a half-disk,
3. the branch point of a cross-shaped set.

For each point, take a very small step in 360 directions and ask whether that nearby point is still in the set. An interior point sees essentially the full circle of directions. A boundary point sees one contiguous half-circle of directions. A branch point sees several separated direction intervals, which is not the local signature of an open disk or a half-disk. This is a computational analogy for checking local Euclidean and boundary charts.


In [ ]:
disk = Point(0, 0).buffer(1.0, resolution=128)
half_disk = disk.intersection(box(-1.05, 0.0, 1.05, 1.05))
cross = unary_union([box(-1.0, -0.12, 1.0, 0.12), box(-0.12, -1.0, 0.12, 1.0)])


def direction_profile(geom, point, eps=0.25, samples=360):
    px, py = point
    angles = np.linspace(0.0, 2 * np.pi, samples, endpoint=False)
    inside = np.array(
        [
            geom.covers(Point(px + eps * np.cos(a), py + eps * np.sin(a)))
            for a in angles
        ],
        dtype=bool,
    )
    if inside.all():
        interval_count = 1
    elif not inside.any():
        interval_count = 0
    else:
        interval_count = int(np.count_nonzero(inside != np.roll(inside, 1)) // 2)
    return angles, inside, {
        "coverage_fraction": float(inside.mean()),
        "direction_interval_count": interval_count,
    }


def draw_polygon(ax, geom, color):
    if geom.geom_type == "Polygon":
        xs, ys = geom.exterior.xy
        ax.fill(xs, ys, color=color, alpha=0.55, linewidth=1.4, edgecolor="#0f172a")
        for ring in geom.interiors:
            rx, ry = ring.xy
            ax.fill(rx, ry, color="white")
    else:
        for part in geom.geoms:
            draw_polygon(ax, part, color)


lab_specs = [
    ("disk interior", disk, (0.0, 0.0), "full disk local model", "#93c5fd"),
    ("half-disk boundary", half_disk, (0.0, 0.0), "half-space local model", "#86efac"),
    ("cross branch", cross, (0.0, 0.0), "not a manifold local model", "#fca5a5"),
]

lab_rows = []
profiles = []
for name, geom, point, expected, color in lab_specs:
    angles, inside, stats = direction_profile(geom, point)
    profiles.append((name, geom, point, expected, color, angles, inside, stats))
    lab_rows.append(
        {
            "probe": name,
            "expected_model": expected,
            "coverage_fraction": round(stats["coverage_fraction"], 4),
            "direction_interval_count": stats["direction_interval_count"],
        }
    )

fig, axes = plt.subplots(3, 2, figsize=(11, 9), gridspec_kw={"width_ratios": [1.05, 1.25]})
for row, (name, geom, point, expected, color, angles, inside, stats) in enumerate(profiles):
    ax_shape = axes[row, 0]
    draw_polygon(ax_shape, geom, color)
    ax_shape.scatter([point[0]], [point[1]], s=70, color="#0f172a", zorder=5)
    ax_shape.set_aspect("equal", adjustable="box")
    ax_shape.set_xlim(-1.15, 1.15)
    ax_shape.set_ylim(-1.15, 1.15)
    ax_shape.set_title(f"{name}: {expected}", fontsize=10)
    ax_shape.set_xticks([])
    ax_shape.set_yticks([])

    ax_bar = axes[row, 1]
    ax_bar.imshow(inside.reshape(1, -1), aspect="auto", cmap=ListedColormap(["#fee2e2", "#166534"]))
    ax_bar.set_yticks([])
    ax_bar.set_xticks([0, 90, 180, 270, 359])
    ax_bar.set_xticklabels(["0", "pi/2", "pi", "3pi/2", "2pi"])
    ax_bar.set_title(
        f"coverage={stats['coverage_fraction']:.2f}; intervals={stats['direction_interval_count']}",
        fontsize=10,
    )
    ax_bar.set_xlabel("sampled direction angle")

fig.suptitle("Applied lab: local direction signatures", fontsize=14)
fig.tight_layout()
lab_png = save_matplotlib(fig, FIGURES / "applied-lab-local-model-direction-test.png")
plt.close(fig)

lab_csv = save_csv(lab_rows, TABLES / "applied-lab-local-model-direction-test.csv")
lab_checks = {
    "rows": lab_rows,
    "interior_has_full_direction_coverage": lab_rows[0]["coverage_fraction"] > 0.95,
    "boundary_has_one_half_space_interval": lab_rows[1]["direction_interval_count"] == 1
    and 0.45 <= lab_rows[1]["coverage_fraction"] <= 0.56,
    "branch_has_multiple_direction_intervals": lab_rows[2]["direction_interval_count"] >= 4,
}
lab_json = save_json(lab_checks, CHECKS / "applied-lab-local-model-direction-test.json")

display(pd.DataFrame(lab_rows))
for artifact in [lab_png, lab_csv, lab_json]:
    display_artifact(artifact)


## Final Sanity Checks

The final cell checks both software artifacts and mathematical invariants. The artifact checks catch stale paths, empty files, and missing visual outputs. The mathematical checks assert the core chapter lessons used in the notebook: finite examples satisfy the topology axioms, Sierpinski is not Hausdorff, the good basis passes the refinement test while the bad basis fails, the basis criterion agrees with full continuity, the circle chart residual is tiny, the closed interval is boundary-only in the gatekeeper matrix, and the local direction lab distinguishes interior, boundary, and branch behavior.


In [ ]:
artifact_paths = [
    storyboard_path,
    finite_png,
    finite_json,
    finite_csv,
    basis_png,
    basis_json,
    basis_csv,
    proof_png,
    proof_json,
    sequence_json,
    atlas_html,
    chart_json,
    gatekeeper_png,
    gatekeeper_csv,
    gatekeeper_json,
    lab_png,
    lab_csv,
    lab_json,
]

assert_artifacts(artifact_paths, min_bytes=64)

assert finite_checks["spaces"]["discrete_3"]["is_topology"]
assert finite_checks["spaces"]["sierpinski"]["is_topology"]
assert not finite_checks["spaces"]["sierpinski"]["hausdorff"]
assert finite_checks["spaces"]["discrete_3"]["local_euclidean_dim_0"]
assert not finite_checks["spaces"]["particular_point"]["hausdorff"]

assert basis_checks["good_basis"]["is_basis"]
assert basis_checks["good_basis"]["generated_is_topology"]
assert not basis_checks["bad_basis"]["is_basis"]
assert basis_checks["continuity"]["continuous_map"]["criterion_agrees"]
assert basis_checks["continuity"]["failing_map"]["criterion_agrees"]
assert basis_checks["continuity"]["continuous_map"]["all_open_preimages"]
assert not basis_checks["continuity"]["failing_map"]["all_open_preimages"]

assert "Hausdorff separation" in proof_graph.nodes
assert "second countable basis" in proof_graph.nodes
assert "local Euclidean charts" in proof_graph.nodes
assert not sequence_checks["indiscrete_3"]["unique_limit_for_this_sequence"]

assert chart_checks["circle_chart_max_inverse_residual"] < 1e-12
assert chart_checks["arc_half_width_less_than_pi"]

assert "closed interval [0,1]" in gatekeeper_checks["boundary_only"]
assert "line with doubled origin" in gatekeeper_checks["non_hausdorff_candidates"]
assert gatekeeper_checks["finite_hausdorff_zero_manifold_matches_discrete"]

assert lab_checks["interior_has_full_direction_coverage"]
assert lab_checks["boundary_has_one_half_space_interval"]
assert lab_checks["branch_has_multiple_direction_intervals"]

final_sanity = {
    "artifact_count_checked_before_final_json": len(artifact_paths),
    "required_png_artifacts": [
        finite_png.name,
        basis_png.name,
        proof_png.name,
        gatekeeper_png.name,
        lab_png.name,
    ],
    "required_json_checks": [
        storyboard_path.name,
        finite_json.name,
        basis_json.name,
        chart_json.name,
        gatekeeper_json.name,
        lab_json.name,
    ],
    "core_invariants": {
        "sierpinski_valid_but_not_hausdorff": finite_checks["spaces"]["sierpinski"]["is_topology"]
        and not finite_checks["spaces"]["sierpinski"]["hausdorff"],
        "good_basis_bad_basis_contrast": basis_checks["good_basis"]["is_basis"]
        and not basis_checks["bad_basis"]["is_basis"],
        "circle_chart_residual": chart_checks["circle_chart_max_inverse_residual"],
        "branch_detected_by_direction_intervals": lab_checks["branch_has_multiple_direction_intervals"],
    },
}
final_sanity_path = save_json(final_sanity, CHECKS / "final-sanity.json")
assert_artifacts([final_sanity_path], min_bytes=64)
display(Markdown("Final sanity checks passed."))
display_artifact(final_sanity_path)


## Takeaways

Topological spaces are controlled by open sets, not by distances. Once open sets are the primitive data, definitions such as closure, interior, continuity, and Hausdorff separation become tests about neighborhoods and preimages.

A basis is a disciplined compression of a topology. The cover condition says every point is locally seen by the basis; the intersection-refinement condition says overlaps are still locally explainable by basis elements. This is why continuity into a space can be checked on a target basis.

Hausdorffness and countability are structural safeguards. Hausdorffness prevents nonunique limits and unseparable points. Second countability gives a global countable supply of basic open sets, strong enough to support first countability, separability, and Lindelof-style countable subcovers.

The manifold definition is a gate, not just a slogan. Locally Euclidean alone is not enough; the doubled-origin model warns that local line charts do not force Hausdorff separation. Hausdorff and second countable alone are not enough; finite non-discrete examples and non-manifold branch points fail local Euclidean behavior. A topological manifold in this chapter is precisely the space that passes all three tests with one fixed local dimension.

Manifolds with boundary use half-space charts. The closed interval is the one-dimensional prototype: interior points look like open intervals, endpoints look like half-lines, and this difference is part of the structure rather than a defect.
